# Plot A: category montages (low to high exemplar variability)

**Purpose:** Build CCN 2026 category montages for the least and most variable categories, with **25 exemplars per montage**.

**BabyView (`bv_*`):** Categories are restricted to the same pool as Plot B (**`data/included_categories.txt` ∩ per-class precision > 0.6** in `annotation/per_class_validation_data.csv`, typically ~85). Only exemplars that pass **`annotation/per_file_precision_data.csv`** (rater validation) are shown. **Mean distance to the BV category centroid** is recomputed from **per-image** embeddings under `clip_embeddings_new` / DINOv3 (filtered metadata rows that match validated stems)—not from the precomputed manuscript variability CSV. **Rank** (optional; see `show_rank_in_montage_text` in Parameters) is the category’s **1-based rank** among the **full Plot B pool** (all categories with a mean-distance estimate, sorted low → high). Ranks are always written to the selection CSV; figure text can omit them when the flag is False.

**THINGS configs:** Still use the manuscript `*_within_category_variability.csv` for ranking; montages use THINGS images (no per-file BV filter).

**Exemplar pool:** `annotation/sampled_object_crops_100_bucket_assignments_100ex_8subj_per_video_cap_babyview_only.csv`, sorted by YOLO confidence within rater-validated rows.

**Output:** montage PNGs/PDFs for categories sampled evenly along the validated low→high variability axis.

## Parameters

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

SCRIPT_DIR = Path(".").resolve()
PROJECT_ROOT = SCRIPT_DIR.parent.parent  # object-detection repo root

# Keep this aligned with your filtered exemplar variability outputs
FILTER_THRESHOLD = 0.27
config = "bv_clip"  # bv_clip | bv_dinov3 | things_clip | things_dinov3

# Manuscript pipeline CSV (used only for non-BV / things_* ranking fallback)
variability_csv = PROJECT_ROOT / "analysis/manuscript-2026/exemplar_variability_analyses/bv_clip_within_category_variability.csv"
grouped_embeddings_dir = None
# Per-image BV embeddings + filtered metadata (same as Plot B) — required to compute validated mean distance to centroid
metadata_csv = PROJECT_ROOT / f"frame_data/merged_frame_detections_with_metadata_filtered-{FILTER_THRESHOLD}.csv"

# High-confidence exemplar pool selected for annotation/figures
sampled_exemplars_csv = PROJECT_ROOT / "annotation/sampled_object_crops_100_bucket_assignments_100ex_8subj_per_video_cap_babyview_only.csv"
semantic_csv = PROJECT_ROOT / "data/long_tailed_dist_prop_included_categories.csv"

cropped_dir = Path(os.getenv("BV_CROPS_BASE", "SET_BV_CROPS_BASE")).expanduser()
things_images_dir = None  # required for things_* configs
out_dir = SCRIPT_DIR / "plotA_category_montages_low_to_high"

# Plot A request: exactly five categories, ranked low -> high variability.
n_selected_categories = 5
# If False: panel titles and output filenames omit rank (CSV still has rank columns).
show_rank_in_montage_text = False
exclude_body_parts = True
BODY_PARTS = {
    "ankle", "arm", "ear", "eye", "face", "finger", "foot", "hair", "hand", "leg",
    "mouth", "neck", "nose", "toe", "tooth",
}

# 5x5 montage per category.
n_exemplars = 25
cell_size = (128, 128)
n_cols = 5

# Combined panel should be horizontal (single row).
combined_n_categories = n_selected_categories
combined_n_cols = n_selected_categories

EXCLUDED_SUBJECT = "00270001"

In [ ]:
GROUPED_EMBEDDINGS_BASE = Path(os.getenv("BV_EMBEDDINGS_BASE", "SET_BV_EMBEDDINGS_BASE")).expanduser()
DEFAULT_GROUPED_EMBEDDINGS = {
    "bv_clip": GROUPED_EMBEDDINGS_BASE / f"clip_embeddings_grouped_by_age-mo_filtered-{FILTER_THRESHOLD}_normalized",
    "bv_dinov3": GROUPED_EMBEDDINGS_BASE / f"dinov3_embeddings_grouped_by_age-mo_filtered-{FILTER_THRESHOLD}_normalized",
}
THINGS_EMBEDDINGS = {
    "things_clip": Path(os.getenv("THINGS_CLIP_NPY_CANDIDATE_2", "SET_THINGS_CLIP_NPY_DIR")).expanduser(),
    "things_dinov3": Path(os.getenv("THINGS_DINOV3_DIR", "SET_THINGS_DINOV3_DIR")).expanduser(),
}

## Helpers: crop dir, THINGS image by index, loaders, montage

In [ ]:
def get_category_crop_dir(cropped_dir, cat_name):
    cat_lower = cat_name.strip().lower()
    direct = cropped_dir / cat_name
    if direct.exists() and direct.is_dir():
        return direct
    for p in cropped_dir.iterdir():
        if p.is_dir() and p.name.lower() == cat_lower:
            return p
    return cropped_dir / cat_name


_things_image_list_cache = {}

def get_things_image_by_index(things_images_dir, cat_name, index):
    key = (Path(things_images_dir), cat_name)
    if key not in _things_image_list_cache:
        cat_dir = get_category_crop_dir(things_images_dir, cat_name)
        paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            paths.extend(cat_dir.glob(ext))
        paths = sorted(paths, key=lambda p: p.name)
        _things_image_list_cache[key] = paths
    paths = _things_image_list_cache[key]
    if index < 0 or index >= len(paths):
        return None
    return paths[index]


def load_valid_exemplar_keys(per_file_precision_csv, threshold=0.6):
    if per_file_precision_csv is None or not Path(per_file_precision_csv).exists():
        return None
    df = pd.read_csv(per_file_precision_csv, usecols=["filename", "class", "precision"])\
        .dropna(subset=["filename", "class", "precision"]).copy()
    df["class"] = df["class"].astype(str).str.strip().str.lower()
    df["precision"] = pd.to_numeric(df["precision"], errors="coerce")
    df = df[df["precision"] > threshold].copy()
    valid = {}
    for fname_raw, cat_raw, _prec in df[["filename", "class", "precision"]].itertuples(index=False, name=None):
        fname = Path(str(fname_raw).strip()).name
        cat = str(cat_raw).strip().lower()
        stem = Path(fname).stem
        remainder = stem
        prefix = f"{cat}_"
        if stem.startswith(prefix):
            parts = stem.split("_", 2)
            if len(parts) == 3:
                remainder = parts[2]
        entry = valid.setdefault(cat, {"full": set(), "remainder": set()})
        entry["full"].add(stem)
        entry["remainder"].add(remainder)
    return valid


def is_valid_exemplar(cat_name, stem, valid_exemplar_keys):
    if valid_exemplar_keys is None:
        return True
    cat = str(cat_name).strip().lower()
    entry = valid_exemplar_keys.get(cat)
    if not entry:
        return False
    s = str(stem).strip()
    if s in entry["full"] or s in entry["remainder"]:
        return True
    base = Path(s).stem
    return (base in entry["full"]) or (base in entry["remainder"])


def load_grouped_embeddings_and_ids(grouped_dir, categories_set, excluded_subject=None, min_exemplars=2, valid_exemplar_keys=None):
    grouped_dir = Path(grouped_dir)
    category_embeddings = {}
    category_exemplar_ids = {}
    for cat_folder in sorted(grouped_dir.iterdir()):
        if not cat_folder.is_dir():
            continue
        cat_name = cat_folder.name
        if categories_set is not None and cat_name not in categories_set:
            continue
        embs, ids = [], []
        for f in cat_folder.glob("*.npy"):
            stem = f.stem
            parts = stem.split("_")
            if len(parts) < 2:
                continue
            subject_id, age_mo = parts[0], None
            try:
                age_mo = int(parts[1])
            except ValueError:
                continue
            if excluded_subject and subject_id == excluded_subject:
                continue
            if not is_valid_exemplar(cat_name, stem, valid_exemplar_keys):
                continue
            try:
                e = np.load(f)
                e = np.asarray(e, dtype=np.float64).flatten()
                embs.append(e)
                ids.append((subject_id, age_mo))
            except Exception:
                continue
        if len(embs) >= min_exemplars:
            category_embeddings[cat_name] = np.array(embs)
            category_exemplar_ids[cat_name] = ids
    return category_embeddings, category_exemplar_ids


def load_things_embeddings_and_ids(embeddings_dir, categories_set, min_exemplars=2):
    from load_things_embeddings import load_things_dinov3_from_dir
    return load_things_dinov3_from_dir(Path(embeddings_dir), allowed_categories=categories_set, min_exemplars=min_exemplars)


def parse_confidence_from_stem(stem):
    parts = str(stem).split("_")
    if len(parts) < 2:
        return np.nan
    try:
        return float(parts[1])
    except ValueError:
        return np.nan


def load_sampled_high_confidence_paths(sampled_csv, categories_set, valid_exemplar_keys=None):
    df = pd.read_csv(sampled_csv)
    df = df.dropna(subset=["category", "path", "stem"]).copy()
    df["category_norm"] = df["category"].astype(str).str.strip().str.lower()
    if categories_set is not None:
        allowed = {c.strip().lower() for c in categories_set}
        df = df[df["category_norm"].isin(allowed)]
    if valid_exemplar_keys is not None:
        df = df[df.apply(lambda r: is_valid_exemplar(r["category_norm"], r["stem"], valid_exemplar_keys), axis=1)]
    df["confidence"] = df["stem"].map(parse_confidence_from_stem)
    df = df.dropna(subset=["confidence"])
    sampled = {}
    for cat, g in df.groupby("category_norm"):
        g = g.sort_values("confidence", ascending=False).reset_index(drop=True)
        sampled[cat] = [(Path(p), float(c)) for p, c in zip(g["path"], g["confidence"])]
    return sampled


def load_filtered_metadata(metadata_csv, excluded_subject=None):
    """CLIP-filtered metadata rows; one row per original_embedding_name (Plot B convention)."""
    df = pd.read_csv(
        metadata_csv,
        usecols=["original_embedding_name", "class_name", "subject_id", "age_mo"],
        dtype={"subject_id": str, "class_name": str},
        na_values=[],
        keep_default_na=False,
    )
    df = df.dropna(subset=["original_embedding_name", "class_name"])
    df["class_name"] = df["class_name"].str.strip().str.lower()
    df["subject_id_norm"] = df["subject_id"].astype(str).str.strip().str.lstrip("S")
    df["age_mo"] = pd.to_numeric(df["age_mo"], errors="coerce").fillna(-1).astype(int)
    if excluded_subject:
        df = df[df["subject_id_norm"] != excluded_subject]
    df = df.drop_duplicates(subset=["original_embedding_name"], keep="first")
    return df


def load_bv_embeddings_validated_per_image(
    embedding_short,
    metadata_csv,
    allowed_categories,
    valid_exemplar_keys,
    excluded_subject,
    embeddings_base,
    min_exemplars=2,
):
    """Load per-crop .npy vectors for metadata rows whose stems pass per-file rater precision (same space as Plot B)."""
    subdir = (
        "clip_embeddings_new"
        if embedding_short == "clip"
        else "facebook_dinov3-vitb16-pretrain-lvd1689m"
    )
    embeddings_dir = Path(embeddings_base) / subdir
    if not embeddings_dir.exists():
        raise FileNotFoundError(f"BV embedding dir not found: {embeddings_dir}")
    meta = load_filtered_metadata(metadata_csv, excluded_subject=excluded_subject)
    meta = meta[meta["class_name"].isin(allowed_categories)]
    category_embeddings = {}
    category_exemplar_ids = {}
    for cat_name, g in meta.groupby("class_name", sort=False):
        cat_folder = get_category_crop_dir(embeddings_dir, cat_name)
        embs, ids = [], []
        for _, row in g.iterrows():
            emb_name = row["original_embedding_name"]
            stem = str(emb_name).rstrip(".npy") if str(emb_name).endswith(".npy") else str(emb_name)
            if not is_valid_exemplar(cat_name, stem, valid_exemplar_keys):
                continue
            path = cat_folder / emb_name
            if not path.exists() and not str(emb_name).endswith(".npy"):
                path = cat_folder / f"{emb_name}.npy"
            if not path.exists():
                continue
            try:
                e = np.load(path)
                e = np.asarray(e, dtype=np.float64).flatten()
                embs.append(e)
                ids.append((row["subject_id_norm"], row["age_mo"], stem))
            except Exception:
                continue
        if len(embs) >= min_exemplars:
            category_embeddings[cat_name] = np.array(embs)
            category_exemplar_ids[cat_name] = ids
    return category_embeddings, category_exemplar_ids


def make_montage(images, n_cols, cell_size):
    if not images:
        return None
    n = len(images)
    n_rows = (n + n_cols - 1) // n_cols
    total_h = n_rows * cell_size[1]
    total_w = n_cols * cell_size[0]
    out = Image.new("RGB", (total_w, total_h), (255, 255, 255))

    for idx, img in enumerate(images):
        row, col = idx // n_cols, idx % n_cols
        if img.size != (cell_size[0], cell_size[1]):
            img = img.resize((cell_size[0], cell_size[1]), Image.Resampling.LANCZOS)
        r0 = row * cell_size[1]
        c0 = col * cell_size[0]
        out.paste(img, (c0, r0))
    return out


def escape_matplotlib_mathtext(s):
    out = str(s).replace("\\", "\\\\")
    out = out.replace("_", r"\_")
    out = out.replace("%", r"\%")
    out = out.replace("#", r"\#")
    out = out.replace("$", r"\$")
    out = out.replace("^", r"\^")
    out = out.replace("{", r"\{")
    out = out.replace("}", r"\}")
    return out


def render_panel_title_math_to_pil(title_mathtext, title_color_rgb, fontsize_pt=20, dpi=200):
    from io import BytesIO

    from matplotlib import mathtext as _mpl_mathtext
    from matplotlib.font_manager import FontProperties

    prop = FontProperties(size=fontsize_pt)
    fc = tuple(c / 255.0 for c in title_color_rgb)
    buf = BytesIO()
    _mpl_mathtext.math_to_image(title_mathtext, buf, dpi=dpi, prop=prop, color=fc)
    buf.seek(0)
    img = Image.open(buf).convert("RGBA")
    bbox = img.getbbox()
    if bbox:
        img = img.crop(bbox)
    return img


def render_panel_title_fit(title_mathtext, title_color_rgb, max_width, max_height, start_fontsize=20, min_fontsize=10, dpi=200):
    size = start_fontsize
    img = render_panel_title_math_to_pil(title_mathtext, title_color_rgb, fontsize_pt=size, dpi=dpi)
    while size > min_fontsize and (img.size[0] > max_width or img.size[1] > max_height):
        size -= 1
        img = render_panel_title_math_to_pil(title_mathtext, title_color_rgb, fontsize_pt=size, dpi=dpi)
    return img

## Category pool, validated BV centroid distances, and selection

In [ ]:
variability_csv = variability_csv or (SCRIPT_DIR / f"{config}_within_category_variability.csv")
out_dir = out_dir or (SCRIPT_DIR / ("exemplar_montages" if config == "bv_clip" else f"exemplar_montages_{config}"))
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

CATEGORIES_FILE = SCRIPT_DIR / "../../data/included_categories.txt"
per_class_validation_csv = SCRIPT_DIR / "../../annotation/per_class_validation_data.csv"
per_file_precision_csv = SCRIPT_DIR / "../../annotation/per_file_precision_data.csv"
precision_threshold = 0.6

if not CATEGORIES_FILE.exists():
    raise FileNotFoundError(f"Required category list: {CATEGORIES_FILE}")
if not per_class_validation_csv.exists():
    raise FileNotFoundError(f"Required for category gate (match Plot B): {per_class_validation_csv}")

with open(CATEGORIES_FILE) as f:
    included_from_file = set(line.strip().lower() for line in f if line.strip())
val_df_gate = pd.read_csv(per_class_validation_csv, usecols=["class", "precision"])\
    .dropna(subset=["class", "precision"]).copy()
val_df_gate["class"] = val_df_gate["class"].astype(str).str.strip().str.lower()
val_df_gate["precision"] = pd.to_numeric(val_df_gate["precision"], errors="coerce")
precision_allowed = set(val_df_gate.loc[val_df_gate["precision"] > precision_threshold, "class"])
included_categories = included_from_file & precision_allowed
print(f"Category pool (included list ∩ per-class precision > {precision_threshold}): {len(included_categories)}")

is_bv_config = config.startswith("bv_")
if is_bv_config:
    if not Path(metadata_csv).exists():
        raise FileNotFoundError(f"Required to compute BV centroid distances: {metadata_csv}")
    if not Path(per_file_precision_csv).exists():
        raise FileNotFoundError(f"Required for rater-validated exemplars: {per_file_precision_csv}")
    valid_exemplar_keys = load_valid_exemplar_keys(per_file_precision_csv, threshold=precision_threshold)
    if not valid_exemplar_keys:
        raise ValueError("Per-file precision CSV produced no valid keys; check filename/class/precision columns.")
    embedding_short = "clip" if "clip" in config else "dinov3"
    cat_emb_valid, _ = load_bv_embeddings_validated_per_image(
        embedding_short,
        metadata_csv,
        included_categories,
        valid_exemplar_keys,
        EXCLUDED_SUBJECT,
        GROUPED_EMBEDDINGS_BASE,
        min_exemplars=2,
    )
    rows = []
    for cat, X in cat_emb_valid.items():
        centroid = X.mean(axis=0)
        mean_d = float(np.mean(np.linalg.norm(X - centroid, axis=1)))
        rows.append({
            "category": cat,
            "mean_dist_to_centroid": mean_d,
            "n_valid_embeddings": int(X.shape[0]),
        })
    var_df = pd.DataFrame(rows)
else:
    valid_exemplar_keys = None
    vcsv = Path(variability_csv)
    if not vcsv.exists():
        raise FileNotFoundError(f"THINGS ranking CSV not found: {vcsv}")
    var_df = pd.read_csv(vcsv)
    var_df["category"] = var_df["category"].astype(str).str.strip().str.lower()
    var_df = var_df[var_df["category"].isin(included_categories)].copy()

# Rank for montage text: full pool sorted by mean distance (matches Plot B category count before montage-only filters).
_var_for_pool_rank = var_df.sort_values("mean_dist_to_centroid", ascending=True).reset_index(drop=True)
n_pool_categories = len(_var_for_pool_rank)
_pool_rank_by_cat = dict(zip(_var_for_pool_rank["category"], range(1, n_pool_categories + 1)))

if exclude_body_parts:
    var_df = var_df[~var_df["category"].isin(BODY_PARTS)].copy()

required_valid_exemplars = n_cols * n_cols
sampled_all_valid = load_sampled_high_confidence_paths(
    sampled_exemplars_csv,
    categories_set=included_categories,
    valid_exemplar_keys=valid_exemplar_keys,
)
valid_counts = {cat: len(v) for cat, v in sampled_all_valid.items()}
var_df["n_valid_exemplars"] = var_df["category"].map(valid_counts).fillna(0).astype(int)
var_df = var_df[var_df["n_valid_exemplars"] >= required_valid_exemplars].copy()

var_df = var_df.sort_values("mean_dist_to_centroid", ascending=True).reset_index(drop=True)
var_df["rank_among_eligible"] = np.arange(1, len(var_df) + 1)
n_eligible = len(var_df)
if n_eligible < n_selected_categories:
    raise ValueError(
        f"Need at least {n_selected_categories} categories with >= {required_valid_exemplars} "
        f"rater-validated exemplars in the sampled CSV; found {n_eligible}."
    )

selected_pos = np.linspace(0, n_eligible - 1, n_selected_categories, dtype=int)
selected_subset = var_df.iloc[selected_pos].copy().reset_index(drop=True)
selected_subset["rank_among_pool"] = selected_subset["category"].map(_pool_rank_by_cat).astype(int)
selected_categories = [
    (
        row["category"],
        i + 1,
        float(row["mean_dist_to_centroid"]),
        int(row["n_valid_exemplars"]),
        int(row["rank_among_eligible"]),
        n_eligible,
        int(row["rank_among_pool"]),
        n_pool_categories,
    )
    for i, row in selected_subset.iterrows()
]
categories_set = set(c[0] for c in selected_categories)

print("Selected categories (panels left→right = low → high validated mean dist to BV centroid):")
for cat_name, panel_i, mean_dist, n_samp, rank_el, n_el, rank_pool, n_pool in selected_categories:
    rank_txt = f" — rank {rank_pool}/{n_pool}" if show_rank_in_montage_text else ""
    print(
        f"  Panel {panel_i}. {cat_name}{rank_txt} | "
        f"mean_dist_to_centroid={mean_dist:.4f} | n_valid_sampled={n_samp}"
    )

## Load embeddings and (for BV) crop lookup

In [ ]:
is_bv = config.startswith("bv_")
print(f"Loading embeddings for {config}...")
if is_bv:
    # Montages use only sampled CSV paths; centroid distances were computed from validated per-image .npy above.
    print("Loading sampled rater-validated exemplars for montages...")
    sampled_by_category = load_sampled_high_confidence_paths(
        sampled_exemplars_csv, categories_set, valid_exemplar_keys=valid_exemplar_keys
    )
    category_embeddings, category_exemplar_ids = {}, {}
    things_images_dir = None
else:
    embeddings_dir = THINGS_EMBEDDINGS[config]
    category_embeddings, category_exemplar_ids = load_things_embeddings_and_ids(
        embeddings_dir, categories_set, min_exemplars=2
    )
    sampled_by_category = None
    things_images_dir = things_images_dir
    if things_images_dir is None or not Path(things_images_dir).exists():
        print("Warning: things_images_dir not set or missing. THINGS montages need images at {dir}/{category}/{stem}.jpg")
        things_images_dir = None

## Build and save montages

In [ ]:
montages_for_combined = []
selected_records = []

semantic_map = {}
if semantic_csv.exists():
    sem_df = pd.read_csv(semantic_csv, usecols=["category", "cdi_semantic"]).dropna(subset=["category", "cdi_semantic"]).copy()
    sem_df["category"] = sem_df["category"].astype(str).str.strip().str.lower()
    sem_df["cdi_semantic"] = sem_df["cdi_semantic"].astype(str).str.strip().str.lower()
    semantic_map = dict(zip(sem_df["category"], sem_df["cdi_semantic"]))

def _cdi_hex_to_rgb255(h):
    h = h.strip()
    if h.startswith("#"):
        h = h[1:]
    return tuple(int(h[i : i + 2], 16) for i in (0, 2, 4))


# Same hexes as Plot B ranked bar + Plot C 2x2 (`CDI_SEMANTIC_COLORS` in notebooks 02 / 03).
_CDI_SEMANTIC_COLORS_HEX = {
    "animals": "#4DB8A8",
    "body_parts": "#E87A5F",
    "clothing": "#9B7EC8",
    "food_drink": "#E8A54C",
    "furniture_rooms": "#6BAB7A",
    "household": "#D97B9E",
    "outside": "#5B9BD5",
    "people": "#E8C44C",
    "toys": "#B07CC8",
    "vehicles": "#6BA3D5",
    "other": "#8B9A9E",
}
semantic_palette = {k: _cdi_hex_to_rgb255(v) for k, v in _CDI_SEMANTIC_COLORS_HEX.items()}
semantic_palette["unknown"] = semantic_palette["other"]

for cat_name, panel_i, mean_dist, n_valid, rank_among_eligible, n_eligible, rank_among_pool, n_pool_categories in tqdm(selected_categories, desc="Montages"):
    if (not is_bv) and (cat_name not in category_embeddings):
        continue

    images = []
    cat_key_lower = cat_name.strip().lower()

    if is_bv:
        exemplars = sampled_by_category.get(cat_key_lower, []) if sampled_by_category is not None else []
        for crop_path, _conf in exemplars[:n_exemplars]:
            if crop_path is None or not Path(crop_path).exists():
                continue
            try:
                img = Image.open(crop_path).convert("RGB")
                images.append(img)
            except Exception:
                continue
    else:
        X = category_embeddings[cat_name]
        centroid = X.mean(axis=0)
        dists = np.linalg.norm(X - centroid, axis=1)
        order = np.argsort(dists)
        n_show = min(n_exemplars, len(order))
        indices = np.linspace(0, len(order) - 1, n_show, dtype=int) if len(order) > n_show else np.arange(len(order))
        selected_idx = order[indices]

        for i in selected_idx:
            if things_images_dir is None:
                continue
            crop_path = get_things_image_by_index(things_images_dir, cat_name, i)
            if crop_path is None or not crop_path.exists():
                continue
            try:
                img = Image.open(crop_path).convert("RGB")
                images.append(img)
            except Exception:
                continue

    if not images:
        continue

    # Enforce 5x5 montage layout; pad with blanks when fewer valid exemplars are available.
    target_n = n_cols * n_cols
    if len(images) < target_n:
        print(f"Warning: {cat_name} has only {len(images)} valid exemplars; padding to {target_n}.")
    images = images[:target_n]
    while len(images) < target_n:
        images.append(Image.new("RGB", cell_size, (245, 245, 245)))

    montage = make_montage(images, n_cols, cell_size)
    if montage is None:
        continue

    if show_rank_in_montage_text:
        rank_label = f"r{rank_among_pool}_of_{n_pool_categories}_p{panel_i:02d}"
    else:
        rank_label = f"p{panel_i:02d}"
    dist_label = f"dist={mean_dist:.4f}"
    out_name = f"exemplar_montage_{rank_label}_{cat_name}_{dist_label}.png"
    out_path = out_dir / out_name
    out_pdf_path = out_path.with_suffix(".pdf")
    montage.save(out_path)
    montage.save(out_pdf_path, "PDF", resolution=300.0)

    semantic_group = semantic_map.get(cat_name, "unknown")
    title_color = semantic_palette.get(semantic_group, semantic_palette["unknown"])
    cat_esc = escape_matplotlib_mathtext(cat_name)
    v_math = rf"$V_{{\mathrm{{c}}}} = {mean_dist:.2f}$"
    if show_rank_in_montage_text:
        panel_title_mt = f"{cat_esc} | rank {rank_among_pool}/{n_pool_categories} | {v_math}"
    else:
        panel_title_mt = f"{cat_esc} | {v_math}"
    montages_for_combined.append((panel_title_mt, title_color, montage.copy()))
    selected_records.append({
        "panel_left_to_right": panel_i,
        "rank_among_eligible": rank_among_eligible,
        "n_eligible_categories": n_eligible,
        "rank_among_pool": rank_among_pool,
        "n_pool_categories": n_pool_categories,
        "category": cat_name,
        "cdi_semantic": semantic_group,
        "n_valid_exemplars": n_valid,
        "mean_dist_to_centroid": mean_dist,
        "montage_file": out_name,
    })

if selected_records:
    selected_df = pd.DataFrame(selected_records).sort_values("panel_left_to_right")
    selected_csv = out_dir / "plotA_selected_categories_low_to_high_variability.csv"
    selected_df.to_csv(selected_csv, index=False)
    print(f"Saved selection metadata: {selected_csv}")

if montages_for_combined:
    # Horizontal strip (5 columns x 1 row).
    panel_cols = len(montages_for_combined)
    panel_rows = 1

    panel_margin = 24
    panel_gap = 24
    panel_title_h = 64

    max_w = max(img.size[0] for _, _, img in montages_for_combined)
    max_h = max(img.size[1] for _, _, img in montages_for_combined)

    panel_w = panel_margin * 2 + panel_cols * max_w + (panel_cols - 1) * panel_gap
    panel_h = panel_margin * 2 + panel_rows * (panel_title_h + max_h)
    combined_panel = Image.new("RGB", (panel_w, panel_h), (255, 255, 255))

    for i, (title_mt, title_color, montage_img) in enumerate(montages_for_combined):
        x0 = panel_margin + i * (max_w + panel_gap)
        y0 = panel_margin

        title_img = render_panel_title_fit(
            title_mt,
            title_color,
            max_width=max_w - 8,
            max_height=panel_title_h - 8,
            start_fontsize=20,
            min_fontsize=10,
            dpi=200,
        )
        text_w, text_h = title_img.size
        tx = x0 + (max_w - text_w) // 2
        ty = y0 + max(0, (panel_title_h - text_h) // 2)
        combined_panel.paste(title_img, (tx, ty), title_img)
        combined_panel.paste(montage_img, (x0, y0 + panel_title_h))

    combined_name = "exemplar_montage_combined_5cats_low_to_high_variability_horizontal.png"
    combined_path = out_dir / combined_name
    combined_pdf_path = combined_path.with_suffix(".pdf")
    combined_svg_path = combined_path.with_suffix(".svg")
    combined_panel.save(combined_path)
    combined_panel.save(combined_pdf_path, "PDF", resolution=300.0)
    # Export SVG via matplotlib so the panel is available in an editable vector container.
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(combined_panel.size[0] / 100, combined_panel.size[1] / 100), dpi=100)
    ax.imshow(combined_panel)
    ax.axis("off")
    fig.savefig(combined_svg_path, format="svg", bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    print(f"Saved combined montage panel: {combined_path}")
    print(f"Saved combined montage panel PDF: {combined_pdf_path}")
    print(f"Saved combined montage panel SVG: {combined_svg_path}")

print(f"Saved montages to {out_dir}")